In [2]:
import ollama

res = ollama.chat(
	model="llava",
	messages=[
		{
			'role': 'user',
			'content': 'Describe this image:',
			'images': ['./art.jpg']
		}
	]
)

print(res['message']['content'])

In [3]:
# ollama.create?x

In [4]:
# modelfile='''
# FROM tinyllama
# SYSTEM You are a helpful chat assistant that uses the given information to do what the user has asked them to do.
# '''

# model = ollama.create(model='assistant-tinyllama', modelfile=modelfile)

In [5]:
# ollama.Client?

In [1]:
from data_loader import DatabaseClient
from ollama import chat, generate

class ChatClient:

    def __init__(self, folder_path = "",  chat_model = 'llama3.1', transform_model = 'phi3'):
        self.messages = []
        self.chat_model = chat_model
        self.transform_model = transform_model
        self.database = DatabaseClient(folder_path=folder_path)
        
    
    def retriever(self, query = None, image_path = None):
        if image_path == None:
            return self.database.search_with_text(query)
        elif query == None:
            return self.database.search_with_image(image_path)
        else:
            return self.database.search_with_text(query).extend(self.database.search_with_image(image_path))  

    def transform_query(self, query):
        content = f"""Extract exactly 3-5 keywords from the following query, and return them in a comma-separated list with no additional text.
        Query: "{query}"
        Keywords: """
        return generate(self.transform_model, prompt=content, options={'temperature':0})['response'

    def handle_properties(self, properties):
        context = ""
        image_paths = []
        for property in properties:
            if property['media_type'] == 'text':
                context += "\n" + property['text']
            if property['media_type'] == 'image':
                image_paths.append(property['path'])
        return context, image_paths

    def user_message(self, content, image_paths):
        return {
            'role' : 'user',
            'content' : content,
            'images' : image_paths
        }

    def input_text(self, user_content):
        transfomed_query = self.transform_query(user_content)
        context, image_paths = self.handle_properties(self.retriever(query = transfomed_query))
        return self.user_message(
            content=f"""Given context : {context}.
                        Do {user_content}""", 
            image_paths=image_paths
            )

    def input_image(self, image_path):
        context, image_paths = self.handle_properties(self.retriever(image_path=image_path))
        return self.user_message(
            content=f"""Explain with help of images. 
                        Context : {context}.""", 
            image_paths=image_paths.append(image_path)
            )
    
    def input_text_image(self, user_content, image_path):
        transfomed_query = self.transform_query(user_content)
        context, image_paths = self.handle_properties(self.retriever(query = transfomed_query, image_path = image_path))
        return self.user_message(
            content=f"""Given context : {context} and images.
                        Do : {user_content}.""",
            image_paths=image_paths.append(image_path)
        )
    
    def interact(self):
        print("Type /exit to end.")
        while True:
            user_content = input("User Content: ")
            user_image_path = input("User Image Path: ")
            if user_content == "/exit":
                self.database.close_connection()
                break
            
            if user_content != '' and user_image_path != '':
                self.messages.append(self.input_text_image(
                    user_content=user_content,
                    image_path=user_image_path
                ))
            elif user_content == '':
                self.messages.append(self.input_text(user_content))
            elif user_image_path == '':
                self.messages.append(self.input_image(user_image_path))
            else:
                continue
            
            assistant_content = ''
            for chunk in chat(self.chat_model, stream=True, messages=self.messages, options={'temperature':0}):
                assistant_content += chunk['message']['content']
                print(chunk['message']['content'], end='', flush=True)
            
            print()
            print('-'*20)
            self.messages.append({
                    'role' : 'assistant',
                    'content' : assistant_content
                })




ImportError: cannot import name 'DatabaseClient' from 'data_loader' (c:\Users\Anush\Desktop\Projects\rag_project\Localinsight\data_loader.py)

In [ ]:

class ChatClient:

    def __init__(self, folder_path = "",  chat_model = 'llama3.1', transform_model = 'phi3'):
        self.messages = []
        self.chat_model = chat_model
        self.transform_model = transform_model
        self.database = DatabaseClient(folder_path=folder_path)
        
    
    def retriever(self, query = None, image_path = None):
        if image_path == None:
            return self.database.search_with_text(query)
        elif query == None:
            return self.database.search_with_image(image_path)
        else:
            return self.database.search_with_text(query).extend(self.database.search_with_image(image_path))
        

    def transform_query(self, query):
        content = f"""Extract exactly 3-5 keywords from the following query, and return them in a comma-separated list with no additional text.

        Query: "{query}"

        Keywords: """

        return generate(self.transform_model, prompt=content, options={'temperature':0})['response']
    
    def input_text(self, user_content):
        transfomed_query = self.transform_query(user_content)
        context = self.handle_properties(self.retriever(query = transfomed_query))
        return f"Given the {context} answer {user_content}"

    def input_image(self, image_path):
        context = self.handle_properties(self.retriever(image_path=image_path))
        return f"Explain the given context : {context}"
    
    def input_text_image(self, user_content, image_path):
        transfomed_query = self.transform_query(user_content)
        context = self.handle_properties(self.retriever(query = transfomed_query, image_path = image_path))

    def handle_properties(self, properties):
        context = ""
        image_paths = []
        for property in properties:
            if property['media_type'] == 'text':
                context += "\n" + property['text']
            if property['media_type'] == 'image':
                image_paths.append(property['path'])
        return context, image_paths

    def user_message(self, content, image_paths):
        return {
            'role' : 'user',
            'content' : content,
            'images' : image_paths
        }
    
    def interact(self):
        print("Type /exit to end.")
        while True:
            user_content = input("User: ")
            if user_content == "/exit":
                self.database.close_connection()
                break
            self.messages.append({
                'role': 'user',
                'content': user_content
            })
            response = chat(self.chat_model, stream=True, messages=self.messages, options={'temperature':0})
            assistant_content = ''
            for chunk in response:
                assistant_content += chunk['message']['content']
                print(chunk['message']['content'], end='', flush=True)
            print()
            print('-'*20)
            self.messages.append({
                    'role' : 'assistant',
                    'content' : assistant_content
                })




In [17]:
chatbot = ChatClient()

In [15]:
chatbot.transform_query(query="What is the use of Machine Learningn in Economics?")

'Machine learning, economics, application, technology, data analysis'

In [134]:
chatbot.interact()

Type /exit to end.
The sky blue color is caused by the presence of various elements in the atmosphere, including:

1. Blue light: This is produced by the sun's ultraviolet rays that reach Earth's surface. The blue light is absorbed by oxygen molecules in the air, which causes the blue color to appear.

2. Water vapor: Water vapor is a major component of the atmosphere and contributes to the blue color of the sky. It absorbs some of the blue light that reaches Earth's surface.

3. Clouds: Clouds absorb some of the blue light, which then reflects back into space. This process creates the blue hue in the sky.

4. Sunlight: The sun's rays also contribute to the blue color of the sky. However, the amount of blue light absorbed by clouds and other factors can vary depending on the time of day, weather conditions, and location.
--------------------
The previous topic was "why is the sky blue?"
--------------------
